In [2]:
import urllib.request
import pandas as pd
import json

base_url = "https://donnees.montreal.ca/api/3/action/datastore_search"
resource_id = "f34c3555-c285-4ef3-a55c-f0f5c440ad2d"
# filters = json.dumps({"categories": "Parcs et espaces verts"})
filters = {}
query_params = urllib.parse.urlencode(
    {"resource_id": resource_id, "filters": filters, "limit": 5000}
)
url = f"{base_url}?{query_params}"
with urllib.request.urlopen(url) as response:
    data_dict = json.loads(response.read().decode('utf-8'))
df = pd.DataFrame(data_dict["result"]["records"])

print(df)

       _id OBJECTID         Type   Lien                                   Nom  \
0        1     3477          NaN    NaN  Université Concordia - Campus Loyola   
1        2     3478          Av.     du                                 Boisé   
2        3     3479         Parc    NaN                        Maurice-Cullen   
3        4     3480          NaN    NaN              Décarie / Zone Tampon #2   
4        5     3481          NaN    NaN              Décarie / Zone Tampon #2   
...    ...      ...          ...    ...                                   ...   
2240  2241     5717   Grand parc  de l'                                   Est   
2241  2242     5718         Parc    NaN                        André-Lavallée   
2242  2243     5719  Parc-nature     du                          Bois-d'Anjou   
2243  2244     5720        Place    des                         Montréalaises   
2244  2245     5721          NaN    NaN                                   NaN   

     NUM_INDEX            S

In [3]:
df = df[df['Nom'].notna()]
df = df[df['COMPETENCE'] != "Privé"]
df

,_id,OBJECTID,Type,Lien,Nom,NUM_INDEX,SUPERFICIE,PROPRIETE,GESTION,COMPETENCE,TYPO1,TYPO2
1,2,3478,Av.,du,Boisé,1641-513,0.03236372202509561,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Espace voirie,Îlot de voirie
2,3,3479,Parc,NaN,Maurice-Cullen,0137-000,0.542680858347123,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Parc d'arrondissement,Parc de quartier
3,4,3480,NaN,NaN,Décarie / Zone Tampon #2,1641-637,0.03351004521482887,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Autre espace vert,Parc écran
4,5,3481,NaN,NaN,Décarie / Zone Tampon #2,1641-637,0.014241730874920712,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Autre espace vert,Parc écran
5,6,3482,Parc,NaN,Trenholme,0158-000,4.711486713378978,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Parc d'arrondissement,Parc de quartier
...,...,...,...,...,...,...,...,...,...,...,...,...
2239,2240,5716,Parc,NaN,Zotique-St-Jean,0991-000,1.2434778966901077,Ville de Montréal,Rivière-des-Prairies-Pointe-aux-Trembles,Arrondissement,Parc d'arrondissement,Îlot de verdure
2240,2241,5717,Grand parc,de l',Est,9514-000,15.930586108176847,Ville de Montréal,"Service des grands parcs, du Mont-Royal et des...",Agglomération,Grand parc,Parc-nature
2241,2242,5718,Parc,NaN,André-Lavallée,0219-000,6.3683783446813536,Ville de Montréal,Rosemont-La Petite-Patrie,Arrondissement,Parc d'arrondissement,Parc de quartier
2242,2243,5719,Parc-nature,du,Bois-d'Anjou,9569-000,52.248061748706434,Ville de Montréal,"Service des grands parcs, du Mont-Royal et des...",Agglomération,Grand parc,Parc-nature


In [4]:
df['full_name'] = df.apply(
    lambda r: (f"{r['Type']} " if pd.notna(r['Type']) else "") + 
              (f"{r['Lien']} " if pd.notna(r['Lien']) else "") + 
              (str(r['Nom']) if pd.notna(r['Nom']) else ""), 
    axis=1
).str.strip()
df['full_name'].unique()

<StringArray>
[                   'Av. du Boisé',             'Parc Maurice-Cullen',
        'Décarie / Zone Tampon #2',                  'Parc Trenholme',
 'Sherbrooke / Décarie, Autoroute',            'Parc Prudence-Heward',
                      'Parc Pilon',                    'Parc Primeau',
         'Parc Victor-T.-Daubigny',         'Parc Carlos-d'Alcantara',
 ...
            'Grand parc de l' Est',            'Parc Arthur-Therrien',
               'Parc de La Presse',             'Parc Francine-Léger',
            'Parc Saidye-Bronfman',          'Passage Yolène-Jumelle',
                 'Damien-Gauthier',            'Parc Zotique-St-Jean',
             'Parc André-Lavallée',         'Place des Montréalaises']
Length: 1451, dtype: str

In [5]:
df['GESTION'].unique()

<StringArray>
[                  'Côte-des-Neiges-Notre-Dame-de-Grâce',
                                           'Ville-Marie',
                                         'Montréal-Nord',
                         'Mercier-Hochelaga-Maisonneuve',
                                   'Pierrefonds-Roxboro',
              'Rivière-des-Prairies-Pointe-aux-Trembles',
                                               'Lachine',
                                         'Saint-Laurent',
                             'Rosemont-La Petite-Patrie',
                                 'Ahuntsic-Cartierville',
                                 'Le Plateau-Mont-Royal',
                                   'Commission scolaire',
                                                 'Autre',
                                                'Verdun',
                                               'LaSalle',
                  'Villeray-Saint-Michel-Parc-Extension',
 'Service des grands parcs, du Mont-Royal et des sports',


In [6]:
not_boroughs = ['Autre', 'Commission scolaire', 'Service des grands parcs, du Mont-Royal et des sports', 'Westmount']
df = df[~df['GESTION'].isin(not_boroughs)]
df = df[df['GESTION'].notna()]
df

,_id,OBJECTID,Type,Lien,Nom,NUM_INDEX,SUPERFICIE,PROPRIETE,GESTION,COMPETENCE,TYPO1,TYPO2,full_name
1,2,3478,Av.,du,Boisé,1641-513,0.03236372202509561,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Espace voirie,Îlot de voirie,Av. du Boisé
2,3,3479,Parc,NaN,Maurice-Cullen,0137-000,0.542680858347123,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Parc d'arrondissement,Parc de quartier,Parc Maurice-Cullen
3,4,3480,NaN,NaN,Décarie / Zone Tampon #2,1641-637,0.03351004521482887,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Autre espace vert,Parc écran,Décarie / Zone Tampon #2
4,5,3481,NaN,NaN,Décarie / Zone Tampon #2,1641-637,0.014241730874920712,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Autre espace vert,Parc écran,Décarie / Zone Tampon #2
5,6,3482,Parc,NaN,Trenholme,0158-000,4.711486713378978,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Parc d'arrondissement,Parc de quartier,Parc Trenholme
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2214,2215,5691,Passage,NaN,Yolène-Jumelle,1332-000,0.22773417485432657,Ville de Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,Arrondissement,Espace voirie,Passage,Passage Yolène-Jumelle
2236,2237,5713,NaN,NaN,Damien-Gauthier,1231-000,2.0133517122702185,Ville de Montréal,Rivière-des-Prairies-Pointe-aux-Trembles,Arrondissement,Parc d'arrondissement,Îlot de verdure,Damien-Gauthier
2239,2240,5716,Parc,NaN,Zotique-St-Jean,0991-000,1.2434778966901077,Ville de Montréal,Rivière-des-Prairies-Pointe-aux-Trembles,Arrondissement,Parc d'arrondissement,Îlot de verdure,Parc Zotique-St-Jean
2241,2242,5718,Parc,NaN,André-Lavallée,0219-000,6.3683783446813536,Ville de Montréal,Rosemont-La Petite-Patrie,Arrondissement,Parc d'arrondissement,Parc de quartier,Parc André-Lavallée


In [7]:
df['full_name'].unique()

<StringArray>
[                   'Av. du Boisé',             'Parc Maurice-Cullen',
        'Décarie / Zone Tampon #2',                  'Parc Trenholme',
 'Sherbrooke / Décarie, Autoroute',            'Parc Prudence-Heward',
                      'Parc Pilon',                    'Parc Primeau',
         'Parc Victor-T.-Daubigny',         'Parc Carlos-d'Alcantara',
 ...
              'Place Emily-Coonan',            'Parc Arthur-Therrien',
               'Parc de La Presse',             'Parc Francine-Léger',
            'Parc Saidye-Bronfman',          'Passage Yolène-Jumelle',
                 'Damien-Gauthier',            'Parc Zotique-St-Jean',
             'Parc André-Lavallée',         'Place des Montréalaises']
Length: 1369, dtype: str

In [8]:
for brgh in df['GESTION'].unique():
    green_area = df[df['GESTION']==brgh]['SUPERFICIE'].astype(float).sum()
    print(f"{brgh}: {green_area:.3f}ha")

Côte-des-Neiges-Notre-Dame-de-Grâce: 84.590ha
Ville-Marie: 64.904ha
Montréal-Nord: 38.084ha
Mercier-Hochelaga-Maisonneuve: 191.263ha
Pierrefonds-Roxboro: 135.724ha
Rivière-des-Prairies-Pointe-aux-Trembles: 210.263ha
Lachine: 84.167ha
Saint-Laurent: 147.244ha
Rosemont-La Petite-Patrie: 97.441ha
Ahuntsic-Cartierville: 205.913ha
Le Plateau-Mont-Royal: 67.475ha
Verdun: 222.487ha
LaSalle: 120.191ha
Villeray-Saint-Michel-Parc-Extension: 89.151ha
Anjou: 62.084ha
Le Sud-Ouest: 187.509ha
L'Île-Bizard-Sainte-Geneviève: 43.282ha
Saint-Léonard: 87.506ha
Outremont: 18.261ha


In [ ]:
total_area = {'Côte-des-Neiges-Notre-Dame-de-Grâce': 2140, # according to https://ville.montreal.qc.ca/pls/portal/docs/PAGE/MTL_STATS_FR/MEDIA/DOCUMENTS/CARTE_POPULATION%20ET%20SUPERFICIE%202021.PDF
'Ville-Marie': 1650,
'Montréal-Nord': 1110,
'Mercier-Hochelaga-Maisonneuve': 2540,
'Pierrefonds-Roxboro': 2710,
'Rivière-des-Prairies-Pointe-aux-Trembles': 4230,
'Lachine': 1770,
'Saint-Laurent': 4280,
'Rosemont-La Petite-Patrie': 1590,
'Ahuntsic-Cartierville': 2420,
'Le Plateau-Mont-Royal': 810,
'Verdun': 970,
'LaSalle': 1630,
'Villeray-Saint-Michel-Parc-Extension': 1650,
'Anjou': 1370,
'Le Sud-Ouest': 1570,
'L\'Île-Bizard-Sainte-Geneviève': 2360,
'Saint-Léonard': 1350,
'Outremont': 390}

for brgh in df['GESTION'].unique():
    green_area = df[df['GESTION']==brgh]['SUPERFICIE'].astype(float).sum()
    print(f"{brgh}: {((green_area / total_area[brgh])*100):.2f}%")

Côte-des-Neiges-Notre-Dame-de-Grâce: 3.95%
Ville-Marie: 3.93%
Montréal-Nord: 3.43%
Mercier-Hochelaga-Maisonneuve: 7.53%
Pierrefonds-Roxboro: 5.01%
Rivière-des-Prairies-Pointe-aux-Trembles: 4.97%
Lachine: 4.76%
Saint-Laurent: 3.44%
Rosemont-La Petite-Patrie: 6.13%
Ahuntsic-Cartierville: 8.51%
Le Plateau-Mont-Royal: 8.33%
Verdun: 22.94%
LaSalle: 7.37%
Villeray-Saint-Michel-Parc-Extension: 5.40%
Anjou: 4.53%
Le Sud-Ouest: 11.94%
L'Île-Bizard-Sainte-Geneviève: 1.83%
Saint-Léonard: 6.48%
Outremont: 4.68%
